In [1]:
# -*- coding: utf-8 -*-
"""
ch10 데이터 집계와 그룹연산 — 강의용 주석/해설 버전

원본 코드 흐름은 유지하면서, 각 단계가 무엇을 보여주는지 "강의자 관점"에서 매우 자세한 주석을 추가했습니다.
수업 중 특정 블록만 선택 실행해도 이해가 되도록, 섹션별 머리말과 핵심 포인트를 덧붙였습니다.

⚠️ 실습 파일 경로
- tips.csv, stock_px.csv 등은 예제 디렉터리("examples/")에 있다고 가정합니다.
- Colab 환경이라면, 해당 파일을 먼저 업로드하거나 경로를 수정하세요.

실습 환경
- Python 3.10+
- pandas 2.x
- numpy, matplotlib
- (일부 섹션) statsmodels
"""

'\nch10 데이터 집계와 그룹연산 — 강의용 주석/해설 버전\n\n원본 코드 흐름은 유지하면서, 각 단계가 무엇을 보여주는지 "강의자 관점"에서 매우 자세한 주석을 추가했습니다.\n수업 중 특정 블록만 선택 실행해도 이해가 되도록, 섹션별 머리말과 핵심 포인트를 덧붙였습니다.\n\n⚠️ 실습 파일 경로\n- tips.csv, stock_px.csv 등은 예제 디렉터리("examples/")에 있다고 가정합니다.\n- Colab 환경이라면, 해당 파일을 먼저 업로드하거나 경로를 수정하세요.\n\n실습 환경\n- Python 3.10+\n- pandas 2.x\n- numpy, matplotlib\n- (일부 섹션) statsmodels\n'

In [2]:
# ---------------------------------------------------------------------
# 0) 공통 설정: 패키지 임포트 & 표/난수 출력 옵션
# ---------------------------------------------------------------------
import numpy as np
import pandas as pd
PREVIOUS_MAX_ROWS = pd.options.display.max_rows  # 기존 옵션 백업
pd.options.display.max_columns = 20
pd.options.display.max_rows = 20
pd.options.display.max_colwidth = 80
np.random.seed(12345)  # 실습마다 동일한 난수 결과를 얻기 위해 고정

import matplotlib.pyplot as plt
plt.rc("figure", figsize=(10, 6))  # 그림 기본 크기
np.set_printoptions(precision=4, suppress=True)  # numpy 출력 자리수/지수표기 억제

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
# ---------------------------------------------------------------------
# 1) 예제 DataFrame 생성: 범주형 키 + 수치형 데이터
#    - key1: 문자열 범주 + 결측치 포함
#    - key2: pandas의 nullable 정수형(Int64) + 결측치 포함
#    - data1, data2: 정규분포 난수
# ---------------------------------------------------------------------
df = pd.DataFrame({
    "key1" : ["a", "a", None, "b", "b", "a", None],
    "key2" : pd.Series([1, 2, 1, 2, 1, None, 1], dtype="Int64"),
    "data1" : np.random.standard_normal(7),
    "data2" : np.random.standard_normal(7)
})

# 포인트: pandas의 Int64는 결측치(NA)를 표현할 수 있는 정수 dtype입니다.
# (numpy int와 다름). groupby(dropna=...)에서 결측 처리 동작 차이를 볼 수 있습니다.

In [ ]:
df  # 확인용

In [ ]:
# ---------------------------------------------------------------------
# 2) Series.groupby(키시리즈)
#    - 단일 키 기준으로 그룹 객체 생성
#    - groupby 자체는 지연 객체; 실제 연산(mean 등)을 호출해야 계산 수행
# ---------------------------------------------------------------------
grouped = df["data1"].groupby(df["key1"])  # data1을 key1 값으로 묶음

In [ ]:
grouped  # <pandas.core.groupby.SeriesGroupBy> 형태의 그룹 객체

In [ ]:
# 그룹별 평균 (결측 key1은 기본적으로 제외됨)
grouped.mean()

In [ ]:
# ---------------------------------------------------------------------
# 3) 다중 키 그룹화: [key1, key2]
#    - MultiIndex로 집계 결과가 반환됨
#    - 이후 unstack() 으로 피벗 형태로 전개 가능
# ---------------------------------------------------------------------
means = df["data1"].groupby([df["key1"], df["key2"]]).mean()

In [ ]:
means  # (key1, key2) MultiIndex Series

In [ ]:
means.unstack()  # 열 축으로 한 단계 펼쳐 보기 (피벗과 유사)

In [ ]:
# ---------------------------------------------------------------------
# 4) 외부 배열/리스트를 key로 사용한 그룹화
#    - 길이가 동일하면 DataFrame의 인덱스에 정렬되어 키로 사용 가능
# ---------------------------------------------------------------------
states = np.array(["OH", "CA", "CA", "OH", "OH", "CA", "OH"])  # 인덱스별 주
years = [2005, 2005, 2006, 2005, 2006, 2005, 2006]  # 인덱스별 연도

In [ ]:
# data1을 (states, years)라는 외부 키로 그룹화하여 평균
df["data1"].groupby([states, years]).mean()

In [ ]:
# ---------------------------------------------------------------------
# 5) DataFrame.groupby: 전체 수치열에 대한 집계 (numeric_only 옵션)
#    - 기본적으로 비수치형은 자동 제외되지만, 버전에 따라 경고/옵션이 다를 수 있음
# ---------------------------------------------------------------------
df.groupby("key1").mean()  # key1별로 수치열 평균

In [ ]:
df.groupby("key2").mean(numeric_only=True)  # nullable Int64 포함 수치열 평균

In [ ]:
df.groupby(["key1", "key2"]).mean()  # 다중 키 집계

In [ ]:
# 그룹 크기(size)와 count의 차이
# - size(): 그룹 내 전체 행 수 (결측 포함 여부는 dropna 설정에 따라 상이)
# - count(): 열별 NA가 아닌 값의 개수

df.groupby(["key1", "key2"]).size()

In [ ]:
df.groupby("key1", dropna=False).size()              # 결측 key1 그룹 포함

In [ ]:
df.groupby(["key1", "key2"], dropna=False).size()   # 결측 키 포함 버전

In [ ]:
# 결측을 제외한 유효값 카운트(열마다 NA 무시)
df.groupby("key1").count()

In [ ]:
# ---------------------------------------------------------------------
# 6) 그룹 이터레이션: (그룹이름, 그룹DataFrame) 튜플 반복
#    - 실무에선 디버깅이나 그룹 단위 커스텀 처리에 유용
# ---------------------------------------------------------------------
for name, group in df.groupby("key1"):
    print(name)
    print(group)

In [ ]:
for (k1, k2), group in df.groupby(["key1", "key2"]):
    print((k1, k2))
    print(group)

In [ ]:
# 그룹들을 사전으로 모아두기 (이후 특정 그룹만 빠르게 참조)
pieces = {name: group for name, group in df.groupby("key1")}
pieces["b"]  # key1 == 'b'인 부분 DataFrame

In [ ]:
# ---------------------------------------------------------------------
# 7) 열 방향(axis="columns") 그룹화: 컬럼 -> 레이블 매핑으로 묶기
#    - 데이터 컬럼을 논리적 카테고리(예: 색상 그룹)로 묶어 집계
# ---------------------------------------------------------------------
# 컬럼으로 사람 5×5 실수형 데이터
people = pd.DataFrame(
    np.random.standard_normal((5, 5)),
    columns=["a", "b", "c", "d", "e"],
    index=["Joe", "Steve", "Wanda", "Jill", "Trey"]
)

In [ ]:
# NA 몇 개 삽입
people.iloc[2:3, [1, 2]] = np.nan
people

In [ ]:
# 컬럼명 -> 그룹명 매핑\mapping = {"a": "red", "b": "red", "c": "blue",
           "d": "blue", "e": "red", "f": "orange"}  # f는 매칭되지 않음

by_column = people.groupby(mapping, axis="columns")
by_column.sum()   # 같은 색 그룹끼리 합계 (행 단위)

In [ ]:
# 매핑을 Series로 만들어도 동일
map_series = pd.Series(mapping)
map_series

In [ ]:
people.groupby(map_series, axis="columns").count()  # NA 무시하고 유효값 개수

In [ ]:
# 함수(len)를 키로 사용해 행 인덱스 길이로 그룹핑 (이름 문자열 길이)
people.groupby(len).sum()

In [ ]:
# 함수 + 리스트(외부 키)로 다중 그룹핑
key_list = ["one", "one", "one", "two", "two"]
people.groupby([len, key_list]).min()

In [ ]:
# ---------------------------------------------------------------------
# 8) 다중 컬럼(MultiIndex Columns)을 가진 DataFrame에서 컬럼 레벨로 그룹핑
# ---------------------------------------------------------------------
columns = pd.MultiIndex.from_arrays([
    ["US", "US", "US", "JP", "JP"],
    [1, 3, 5, 1, 3]
], names=["cty", "tenor"])

hier_df = pd.DataFrame(np.random.standard_normal((4, 5)), columns=columns)
hier_df

In [ ]:
# 컬럼 레벨(level="cty") 기준으로 그룹 크기 집계
hier_df.groupby(level="cty", axis="columns").count()

In [ ]:
# ---------------------------------------------------------------------
# 9) 그룹별 사용자 정의 집계 함수 (agg)
# ---------------------------------------------------------------------
df

In [ ]:
grouped = df.groupby("key1")

In [ ]:
# nsmallest: 각 그룹 내에서 가장 작은 n개 선택 (SeriesGroupBy 전용)
grouped["data1"].nsmallest(2)

In [ ]:
# 사용자 정의 범위 함수

def peak_to_peak(arr):
    """최댓값 - 최솟값 (범위)
    numpy 배열/Series 모두 동작
    """
    return arr.max() - arr.min()

In [ ]:
# agg에 사용자 정의 함수 전달
grouped.agg(peak_to_peak)

In [ ]:
# 기본 통계 요약
grouped.describe()

In [ ]:
# ---------------------------------------------------------------------
# 10) 실전 예제: 팁 데이터(tips.csv)
#     - 그룹별 비율/통계, 컬럼별 서로 다른 집계, 사용자 정의 함수 이름 지정
# ---------------------------------------------------------------------
# ⚠️ tips.csv 경로를 확인하세요.
tips = pd.read_csv("examples/tips.csv")
tips.head()

In [ ]:
# 팁 비율 열 추가
tips["tip_pct"] = tips["tip"] / tips["total_bill"]
tips.head()

In [ ]:
# 요일×흡연여부 그룹핑
grouped = tips.groupby(["day", "smoker"])

In [ ]:
# 단일 컬럼의 그룹별 평균
grouped_pct = grouped["tip_pct"]
grouped_pct.agg("mean")

In [ ]:
# 여러 집계 동시에
grouped_pct.agg(["mean", "std", peak_to_peak])

In [ ]:
# (별칭, 함수) 튜플로 칼럼명 제어
grouped_pct.agg([("average", "mean"), ("stdev", np.std)])

In [ ]:
# DataFrame에 대해 컬럼별 서로 다른 함수 적용
functions = ["count", "mean", "max"]
result = grouped[["tip_pct", "total_bill"]].agg(functions)
result

In [ ]:
result["tip_pct"]  # 다중 인덱스 칼럼 중 일부만 선택

In [ ]:
ftuples = [("Average", "mean"), ("Variance", np.var)]
grouped[["tip_pct", "total_bill"]].agg(ftuples)

In [ ]:
# dict 형태로 컬럼별 지정
grouped.agg({"tip" : np.max, "size" : "sum"})

In [ ]:
grouped.agg({"tip_pct" : ["min", "max", "mean", "std"],
             "size" : "sum"})

In [ ]:
# as_index=False: 그룹키를 인덱스가 아니라 일반 열로 유지
grouped = tips.groupby(["day", "smoker"], as_index=False)
grouped.mean(numeric_only=True)

In [ ]:
# ---------------------------------------------------------------------
# 11) groupby + apply: 그룹 내부 정렬 후 상위 N개 추출 함수
# ---------------------------------------------------------------------

def top(df, n=5, column="tip_pct"):
    """주어진 column 기준 내림차순 상위 n개 행 반환"""
    return df.sort_values(column, ascending=False)[:n]

top(tips, n=6)  # 전체에서 상위 6개

In [ ]:
# 그룹별 상위 N개
tips.groupby("smoker").apply(top)  # smoker 그룹별 tip_pct 상위 5개

In [ ]:
tips.groupby(["smoker", "day"]).apply(top, n=1, column="total_bill")  # 각 그룹 1개

In [ ]:
# describe를 groupby 후 적용 -> 다층 열
result = tips.groupby("smoker")["tip_pct"].describe()
result

In [ ]:
result.unstack("smoker")  # 한 축으로 펼치기

In [ ]:
# group_keys=False: apply 결과에 그룹 키를 인덱스 레벨로 붙이지 않음
tips.groupby("smoker", group_keys=False).apply(top)

In [ ]:
# ---------------------------------------------------------------------
# 12) 구간화(cut/qcut) + groupby: 구간별 통계
# ---------------------------------------------------------------------
frame = pd.DataFrame({
    "data1": np.random.standard_normal(1000),
    "data2": np.random.standard_normal(1000)
})
frame.head()

In [ ]:
# 등간 간격 4구간으로 나누기
quartiles = pd.cut(frame["data1"], 4)
quartiles.head(10)

In [ ]:
# 그룹별 통계 DataFrame 반환 함수

def get_stats(group):
    return pd.DataFrame({
        "min": group.min(),
        "max": group.max(),
        "count": group.count(),
        "mean": group.mean()
    })

In [ ]:
# data2를 quartiles 기준으로 그룹핑 후 통계
# (여기서는 group 전체를 Series로 넘기기 위해 frame.groupby(quartiles)["data2"] 로도 가능)
grouped = frame.groupby(quartiles)
grouped.apply(get_stats)

grouped.agg(["min", "max", "count", "mean"])  # 동일 결과를 agg로도 가능

In [ ]:
# 분위수 기준 4분위(qcut). labels=False로 0~3 레이블 반환
quartiles_samp = pd.qcut(frame["data1"], 4, labels=False)
quartiles_samp.head()

In [ ]:
# (0,1,2,3) 분위 그룹별 통계
grouped = frame.groupby(quartiles_samp)
grouped.apply(get_stats)

In [ ]:
# ---------------------------------------------------------------------
# 13) 그룹별 결측치 처리 전략: 평균 대치 / 그룹 고정값 대치
# ---------------------------------------------------------------------
s = pd.Series(np.random.standard_normal(6))
s[::2] = np.nan  # 짝수 위치 결측
s

In [ ]:
s.fillna(s.mean())  # 전체 평균으로 대치 (그룹 사용 안 함)

In [ ]:
# 지역-권역 예제
states = ["Ohio", "New York", "Vermont", "Florida",
          "Oregon", "Nevada", "California", "Idaho"]

group_key = ["East", "East", "East", "East",
             "West", "West", "West", "West"]

data = pd.Series(np.random.standard_normal(8), index=states)
data

In [ ]:
# 일부 주만 결측 만들기
data[["Vermont", "Nevada", "Idaho"]] = np.nan

data

In [ ]:
data.groupby(group_key).size()   # 그룹 전체 크기

In [ ]:
data.groupby(group_key).count()  # 유효값 개수 (NA 제외)

In [ ]:
data.groupby(group_key).mean()   # 그룹 평균

In [ ]:
# 각 그룹 평균으로 결측 대치

def fill_mean(group):
    return group.fillna(group.mean())

In [ ]:
data.groupby(group_key).apply(fill_mean)

In [ ]:
# 그룹 고정값 dict로 대치
fill_values = {"East": 0.5, "West": -1}

In [ ]:
def fill_func(group):
    return group.fillna(fill_values[group.name])  # group.name == 그룹 키

In [ ]:
data.groupby(group_key).apply(fill_func)

In [ ]:
# ---------------------------------------------------------------------
# 14) 카드 덱 예제: groupby + sample + 사용자 키 함수
# ---------------------------------------------------------------------
suits = ["H", "S", "C", "D"]  # Hearts, Spades, Clubs, Diamonds
card_val = (list(range(1, 11)) + [10] * 3) * 4  # A~10,J,Q,K (J/Q/K=10) × 4무늬
base_names = ["A"] + list(range(2, 11)) + ["J", "K", "Q"]

In [ ]:
# 52장 카드 인덱스 생성
decks = []
for suit in suits:
    decks.extend(str(num) + suit for num in base_names)

deck = pd.Series(card_val, index=decks)

deck.head(13)

In [ ]:
# 무작위로 n장 뽑기

def draw(deck, n=5):
    return deck.sample(n)

In [ ]:
draw(deck)

In [ ]:
# 마지막 글자(무늬)로 그룹을 구분하는 키 함수

def get_suit(card):
    return card[-1]

In [ ]:
# 각 무늬 그룹에서 2장씩 샘플
deck.groupby(get_suit).apply(draw, n=2)

In [ ]:
# group_keys=False: 결과 인덱스에 그룹 키 레벨을 추가하지 않음
deck.groupby(get_suit, group_keys=False).apply(draw, n=2)

In [ ]:
# ---------------------------------------------------------------------
# 15) 가중평균 예제: np.average + groupby.apply
# ---------------------------------------------------------------------
df = pd.DataFrame({
    "category": ["a", "a", "a", "a", "b", "b", "b", "b"],
    "data": np.random.standard_normal(8),
    "weights": np.random.uniform(size=8)
})

In [ ]:
grouped = df.groupby("category")

In [ ]:
def get_wavg(group):
    return np.average(group["data"], weights=group["weights"])

In [ ]:
grouped.apply(get_wavg)

In [ ]:
# ---------------------------------------------------------------------
# 16) 주가 수익률 상관 예제: 연도별로 SPX와 개별종목 상관, AAPL-MSFT 상관
# ---------------------------------------------------------------------
# ⚠️ stock_px.csv 경로 확인
close_px = pd.read_csv("examples/stock_px.csv", parse_dates=True, index_col=0)
close_px.info()

In [ ]:
close_px.tail(4)

In [ ]:
# 일별 수익률
trets = close_px.pct_change().dropna()

In [ ]:
# 인덱스가 DatetimeIndex이므로 연도 추출 함수 정의

def get_year(x):
    return x.year

In [ ]:
by_year = trets.groupby(get_year)

In [ ]:
# 각 연도 내에서 모든 열을 SPX 열과 상관관계 계산
def spx_corr(group):
    return group.corrwith(group["SPX"])  # 같은 길이의 Series 간 상관

In [ ]:
by_year.apply(spx_corr)

In [ ]:
# 각 연도별로 AAPL과 MSFT의 상관만 계산

def corr_aapl_msft(group):
    return group["AAPL"].corr(group["MSFT"])  # 두 Series 상관계수

In [ ]:
by_year.apply(corr_aapl_msft)

In [ ]:
# ---------------------------------------------------------------------
# 17) (선택) 간단 회귀: 연도별 AAPL ~ SPX 단순선형회귀 (statsmodels 필요)
# ---------------------------------------------------------------------
import statsmodels.api as sm

In [ ]:
def regress(data, yvar=None, xvars=None):
    Y = data[yvar].copy()
    X = data[xvars].copy()
    X["intercept"] = 1.0  # 상수항 추가
    result = sm.OLS(Y, X).fit()
    return result.params  # 기울기/절편 등 회귀계수

In [ ]:
by_year.apply(regress, yvar="AAPL", xvars=["SPX"])  # 연도별 계수 변화 관찰

In [ ]:
# ---------------------------------------------------------------------
# 18) transform vs apply: 그룹별 표준화/랭크 예제
#     - transform: 원래 인덱스/길이를 보전하며 그룹별 변환 결과를 반환
#     - apply: 반환 형태가 자유롭지만, 종종 인덱스가 달라질 수 있음
# ---------------------------------------------------------------------

df = pd.DataFrame({'key': ['a', 'b', 'c'] * 4,
                   'value': np.arange(12., dtype=float)})

In [ ]:
g = df.groupby('key')['value']

In [ ]:
g.mean()  # 각 그룹 평균

In [ ]:
# 사용자 정의 평균 함수

def get_mean(group):
    return group.mean()

In [ ]:
g.transform(get_mean)  # 각 원소 위치에 그룹 평균을 브로드캐스팅

In [ ]:
g.transform('mean')    # 문자열로도 가능 (내장 함수 명)

In [ ]:
# 각 원소를 두 배로 (그룹별 X 2)

def times_two(group):
    return group * 2

In [ ]:
g.transform(times_two)

In [ ]:
# 그룹 내 내림차순 랭크

def get_ranks(group):
    return group.rank(ascending=False)

In [ ]:
g.transform(get_ranks)

In [ ]:
# 그룹 표준화

def normalize(x):
    return (x - x.mean()) / x.std()

In [ ]:
g.transform(normalize)

In [ ]:
g.apply(normalize)  # 여기서는 transform과 동일 결과 (스칼라가 아닌 1:1 변환)

In [ ]:
# 그룹 평균/표준편차를 활용한 수동 표준화
g_mean = g.transform('mean')
g_std = g.transform('std')
normalized = (df['value'] - g_mean) / g_std
normalized

In [ ]:
# ---------------------------------------------------------------------
# 19) pivot_table: 피벗 테이블로 통계 요약 만들기
# ---------------------------------------------------------------------

tips.head()

In [ ]:
# 인덱스(행), 값(열)을 명시하여 통계 생성
tips.pivot_table(index=["day", "smoker"],
                 values=["size", "tip", "tip_pct", "total_bill"])

In [ ]:
# columns 인자에 범주를 주어 2차원 교차표 형태
tips.pivot_table(index=["time", "day"], columns="smoker",
                 values=["tip_pct", "size"])

In [ ]:
# margins=True: 전체 합(또는 평균) 행/열 추가
tips.pivot_table(index=["time", "day"], columns="smoker",
                 values=["tip_pct", "size"], margins=True)

In [ ]:
# aggfunc=len: 건수 집계 (GroupBy.size와 유사)
tips.pivot_table(index=["time", "smoker"], columns="day",
                 values="tip_pct", aggfunc=len, margins=True)

In [ ]:
# 다중 인덱스 + 결측 대체(fill_value)
tips.pivot_table(index=["time", "size", "smoker"], columns="day",
                 values="tip_pct", fill_value=0)

In [ ]:
# ---------------------------------------------------------------------
# 20) crosstab: 범주형 교차표(빈도표)
# ---------------------------------------------------------------------
from io import StringIO

In [ ]:
data_txt = """Sample  Nationality  Handedness
1   USA  Right-handed
2   Japan    Left-handed
3   USA  Right-handed
4   Japan    Right-handed
5   Japan    Left-handed
6   Japan    Right-handed
7   USA  Right-handed
8   USA  Left-handed
9   Japan    Right-handed
10  USA  Right-handed"""

In [ ]:
data = pd.read_table(StringIO(data_txt), sep="\s+")

In [ ]:
data

In [ ]:
# 단일 축 vs. 다중 축 기준 교차표
pd.crosstab(data["Nationality"], data["Handedness"], margins=True)

In [ ]:
pd.crosstab([tips["time"], tips["day"]], tips["smoker"], margins=True)

In [ ]:
# ---------------------------------------------------------------------
# 21) 마무리: 출력 옵션 원복
# ---------------------------------------------------------------------
pd.options.display.max_rows = PREVIOUS_MAX_ROWS